# Affect-Aware Personalized Ads from Turkish Tweets — MVP (Groq + Drive)

**Author:** Musa Yüksel — 21050911018
**Course:** Neural Networks — Ankara Yıldırım Beyazıt University

End-to-end pipeline:
1. **Emotion data** — load Turkish tweet emotion dataset (5 classes, ~3.9k tweets after dedup)
2. **EDA** — class balance, length stats, top tokens
3. **BERTurk** — fine-tune `dbmdz/bert-base-turkish-cased` on the emotion task (manual PyTorch loop). **Best checkpoint is saved directly to your Google Drive** so it survives Colab session timeouts.
4. **User bundles** — synthesize 25 users (5 balanced + 20 emotion-skewed) from the tweet pool
5. **User vectors** — run BERTurk over each bundle, aggregate per user (mean softmax + TF-IDF keywords)
6. **Persona templates** — render Turkish-language persona blurbs in 3 variants (full / random / generic) for ablation
7. **Product catalog** — 10 fictional Turkish products spanning categories
8. **LLM ad generator** — **Groq (Llama-3.3-70B-Versatile) only**; stub fallback when no key is set
9. **Sweep** — 5 emotions × 2 products × 3 variants = 30 ads
10. **Evaluation** — Layer-2 programmatic metrics + Layer-3 LLM-as-judge

> Get a free Groq API key at [console.groq.com/keys](https://console.groq.com/keys) and add it as a Colab Secret named `GROQ_API_KEY`. §10 detects it automatically. If no key is set the notebook falls back to a deterministic stub so it still runs end-to-end.

> **All outputs (including the trained BERTurk + XLM-R checkpoints) are written to `/content/drive/MyDrive/outputs/`** on Colab, so they persist across runtime restarts.

## §0 Setup

In [ ]:
# Install dependencies (Colab usually has torch + sklearn already; we still
# pin transformers and the Groq SDK).
%pip install -q "transformers>=4.40" "scikit-learn>=1.3" "groq>=0.13" sentencepiece protobuf 2>&1 | tail -3

In [ ]:
# Mount Google Drive on Colab so outputs (including the trained BERTurk +
# XLM-R checkpoints) survive runtime restarts. Falls back to a local folder
# when not on Colab.
import os

DRIVE_FOLDER = 'outputs'  # under /content/drive/MyDrive/

try:
    from google.colab import drive  # type: ignore
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')   # OAuth prompt the first time
    OUT_ROOT = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
    print(f'✓ Drive mounted — outputs will persist to {OUT_ROOT}')
except (ImportError, ModuleNotFoundError):
    OUT_ROOT = './outputs'
    print(f'Not on Colab — using local folder {OUT_ROOT}')

os.makedirs(OUT_ROOT, exist_ok=True)

In [ ]:
# Imports + device + seed.
import json, os, re, time, random, hashlib, math
from collections import Counter, defaultdict
from pathlib import Path
from statistics import mean

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

USE_AMP = (DEVICE.type == 'cuda')
print(f'device = {DEVICE}  (amp = {USE_AMP})')

OUT = Path(OUT_ROOT)
OUT.mkdir(parents=True, exist_ok=True)

## §1 Emotion dataset (Step 1)

Source: [`anilguven/turkish_tweet_emotion_dataset`](https://huggingface.co/datasets/anilguven/turkish_tweet_emotion_dataset) (Güven, Diri & Çakaloğlu 2019).
The HuggingFace auto-loader fails on this CSV (no header row), so we read with `pandas` directly. We also drop ~95 duplicate texts that would otherwise leak between train/test splits.

In [ ]:
EMOTION_TR2EN = {
    "kizgin":  "angry",
    "korku":   "fear",
    "mutlu":   "happy",
    "surpriz": "surprise",
    "uzgun":   "sad",
}
EMOTION_LABELS = ["angry", "fear", "happy", "surprise", "sad"]
EMOTION_LABEL2ID = {l: i for i, l in enumerate(EMOTION_LABELS)}
EMOTION_ID2LABEL = {i: l for l, i in EMOTION_LABEL2ID.items()}

CSV_URL = (
    "https://huggingface.co/datasets/anilguven/turkish_tweet_emotion_dataset"
    "/resolve/main/Turkish_Tweet_Dataset.csv"
)


def load_emotion_dataset(csv_url: str = CSV_URL,
                         drop_duplicates: bool = True) -> pd.DataFrame:
    df = pd.read_csv(csv_url, header=None, names=["text", "label_tr"])
    df = df.dropna(subset=["text", "label_tr"]).reset_index(drop=True)
    df["text"]     = df["text"].astype(str).str.strip()
    df["label_tr"] = df["label_tr"].astype(str).str.strip()
    df = df[df["label_tr"].isin(EMOTION_TR2EN)].reset_index(drop=True)
    if drop_duplicates:
        df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
    df["label"]    = df["label_tr"].map(EMOTION_TR2EN)
    df["label_id"] = df["label"].map(EMOTION_LABEL2ID)
    return df


df_emo = load_emotion_dataset()
print(f"rows: {len(df_emo):,}")
print(f"classes: {df_emo['label'].value_counts().to_dict()}")
df_emo.head()

## §2 EDA (Step 2)

Quick read of the dataset: the corpus is short, balanced, and pre-cleaned (no URLs / mentions / hashtags). p95 length is **12 words** so a BERT `max_length=64` covers >95% of tweets without truncation.

In [ ]:
TR_STOPWORDS = {
    "ve","bir","bu","da","de","için","ile","mi","mı","mu","mü","ne","ya","ki",
    "ben","sen","o","biz","siz","onlar","ama","çok","daha","en","her","şey",
    "var","yok","ise","gibi","kadar","sonra","önce","diye","şu","şimdi",
    "böyle","öyle","hem","hep","hiç","tüm","bütün","az","fakat","ancak",
    "veya","değil","olarak","üzere","oldu","olur","olan","olmak","iken",
    "nasıl","neden","niçin","kim","hangi",
}
URL_RE     = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#\w+")
TOKEN_RE   = re.compile(r"[a-zçğıöşüâîû]+", flags=re.IGNORECASE)


def tokenize_for_eda(text: str) -> list:
    s = URL_RE.sub(" ", text)
    s = MENTION_RE.sub(" ", s)
    s = HASHTAG_RE.sub(" ", s)
    return [t for t in TOKEN_RE.findall(s.lower())
            if len(t) > 2 and t not in TR_STOPWORDS]


df = df_emo.copy()
df["n_chars"] = df["text"].str.len()
df["n_words"] = df["text"].str.split().map(len)

print("class balance:")
counts = df["label"].value_counts().reindex(EMOTION_LABELS)
for c, n in counts.items():
    print(f"  {c:8s} {n}")

print(f"\nmedian length: {int(df['n_words'].median())} words / "
      f"{int(df['n_chars'].median())} chars")
print(f"p95   length: {int(df['n_words'].quantile(0.95))} words / "
      f"{int(df['n_chars'].quantile(0.95))} chars")

top_per_class = {}
for cls in EMOTION_LABELS:
    c = Counter()
    for t in df.loc[df["label"] == cls, "text"]:
        c.update(tokenize_for_eda(t))
    top_per_class[cls] = c.most_common(8)
print("\ntop tokens per class:")
for cls, items in top_per_class.items():
    print(f"  [{cls:8s}] {', '.join(w for w,_ in items)}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(counts.index, counts.values, color="#4C78A8")
axes[0].set_title("class balance"); axes[0].set_ylabel("# tweets")
axes[1].hist(df["n_words"], bins=40, color="#54A24B", edgecolor="white")
axes[1].axvline(df["n_words"].quantile(0.95), color="red", ls="--",
                label=f"p95 = {int(df['n_words'].quantile(0.95))}")
axes[1].set_title("tweet length (words)"); axes[1].legend()
plt.tight_layout(); plt.savefig(OUT/"eda_emotion.png", dpi=130, bbox_inches="tight")
plt.show()

## §3 Classical baselines (rubric §3.3)

Before reaching for a 110M-parameter Transformer, we establish how far classical text classification can get us on this dataset. The comparison answers a real question: how much of BERTurk's accuracy is the Transformer actually contributing vs the dataset being easy?

Two baselines, both on TF-IDF features (word 1-2 grams, sublinear TF, min_df=2):
- **Logistic Regression** (L2, C=1.0)
- **Linear SVM** (one-vs-rest, hinge loss, C=1.0)

We compute the 80/10/10 stratified split here once and reuse it in §4 BERTurk so the comparison is on the **exact same** test set.

In [ ]:
# Shared 80/10/10 stratified splits — reused by §3 baselines and §4 BERTurk.
from sklearn.model_selection import train_test_split
TEST_FRAC = 0.10
VAL_FRAC  = 0.10  # of the full set, so train = 80%

df_trainval, df_test = train_test_split(
    df_emo, test_size=TEST_FRAC,
    stratify=df_emo["label_id"], random_state=SEED)
df_train, df_val = train_test_split(
    df_trainval, test_size=VAL_FRAC / (1 - TEST_FRAC),
    stratify=df_trainval["label_id"], random_state=SEED)
print(f"splits: train={len(df_train)}  val={len(df_val)}  test={len(df_test)}")

In [ ]:
# Train + evaluate two classical baselines on the same split.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, accuracy_score)

OUT_BASE = OUT / "baselines"; OUT_BASE.mkdir(parents=True, exist_ok=True)

vec_base = TfidfVectorizer(
    lowercase=True, token_pattern=r"[a-zçğıöşüâîû]+",
    ngram_range=(1, 2), min_df=2, max_features=50000,
    sublinear_tf=True,
)
X_train = vec_base.fit_transform(df_train["text"])
X_test  = vec_base.transform(df_test["text"])
y_train = df_train["label_id"].values
y_test  = df_test["label_id"].values
print(f"TF-IDF features: {X_train.shape[1]:,}")

baseline_results = {}

# --- TF-IDF + Logistic Regression ---
logreg = LogisticRegression(C=1.0, max_iter=2000, n_jobs=-1)
logreg.fit(X_train, y_train)
y_pred_lr = logreg.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr  = f1_score(y_test, y_pred_lr, average="macro")
print(f"\n=== TF-IDF + Logistic Regression ===")
print(f"test acc={acc_lr:.4f}  macro_f1={f1_lr:.4f}")
print(classification_report(y_test, y_pred_lr,
      target_names=EMOTION_LABELS, digits=4))
baseline_results["logreg"] = {
    "acc": float(acc_lr), "macro_f1": float(f1_lr),
    "per_class_f1": f1_score(y_test, y_pred_lr, average=None).tolist(),
}

# --- TF-IDF + Linear SVM ---
svm = LinearSVC(C=1.0, max_iter=3000)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
acc_svm = accuracy_score(y_test, y_pred_svm)
f1_svm  = f1_score(y_test, y_pred_svm, average="macro")
print(f"\n=== TF-IDF + Linear SVM ===")
print(f"test acc={acc_svm:.4f}  macro_f1={f1_svm:.4f}")
print(classification_report(y_test, y_pred_svm,
      target_names=EMOTION_LABELS, digits=4))
baseline_results["svm"] = {
    "acc": float(acc_svm), "macro_f1": float(f1_svm),
    "per_class_f1": f1_score(y_test, y_pred_svm, average=None).tolist(),
}

# Confusion matrices side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (name, y_pred) in zip(axes, [("LogReg", y_pred_lr),
                                      ("LinearSVC", y_pred_svm)]):
    cm = confusion_matrix(y_test, y_pred,
                          labels=list(range(len(EMOTION_LABELS))))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(EMOTION_LABELS)))
    ax.set_yticks(range(len(EMOTION_LABELS)))
    ax.set_xticklabels(EMOTION_LABELS, rotation=45, ha="right")
    ax.set_yticklabels(EMOTION_LABELS)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    f1v = f1_score(y_test, y_pred, average="macro")
    ax.set_title(f"{name}  acc={accuracy_score(y_test, y_pred):.3f}  "
                 f"f1={f1v:.3f}")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black",
                    fontsize=9)
plt.tight_layout()
plt.savefig(OUT_BASE / "confusion_matrices.png", dpi=130, bbox_inches="tight")
plt.show()

with open(OUT_BASE / "results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)
print(f"\nSaved baseline results → {OUT_BASE}/results.json")

## §4 BERTurk emotion classifier

Manual PyTorch loop: AdamW + linear warmup (10%) + linear decay, gradient clipping at 1.0, AMP only on CUDA, early stopping on val macro-F1 with patience 2.

We reuse the **same train/val/test split** computed in §3 so the head-to-head comparison with the classical baselines is exact.

**Important MPS fix in `evaluate()`** — Apple Silicon's MPS backend silently corrupts a small fraction of `int64` tensor elements when transferred MPS→CPU. Symptom: macro-F1 stuck at ~0.20 even though predictions are correct (sklearn auto-detects garbage label values as phantom classes). Fix: keep label tensors on CPU and only send them to MPS for the loss forward pass. The fix is a no-op on CUDA.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, accuracy_score)
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)


class CFG:
    model_name   = "dbmdz/bert-base-turkish-cased"
    max_length   = 64
    batch_size   = 16
    eval_bs      = 32
    lr           = 2e-5
    weight_decay = 0.01
    epochs       = 6
    warmup_frac  = 0.10
    grad_clip    = 1.0
    patience     = 2
    test_frac    = 0.10
    val_frac     = 0.10
    out_dir      = OUT / "berturk_emotion"


cfg = CFG(); cfg.out_dir.mkdir(parents=True, exist_ok=True)


class EmotionDS(Dataset):
    def __init__(self, texts, labels, tok, max_length):
        self.texts = list(texts); self.labels = list(labels)
        self.tok = tok; self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True,
                       max_length=self.max_length,
                       padding="max_length", return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": torch.tensor(self.labels[i], dtype=torch.long)}


@torch.no_grad()
def evaluate(model, loader, device):
    """MPS-safe eval: labels never round-trip through MPS for metrics."""
    model.eval()
    all_pred, all_true, total_loss, n = [], [], 0.0, 0
    for batch in loader:
        labels_cpu = batch["labels"]
        inputs = {
            "input_ids":      batch["input_ids"].to(device, non_blocking=True),
            "attention_mask": batch["attention_mask"].to(device, non_blocking=True),
            "labels":         labels_cpu.to(device, non_blocking=True),
        }
        out = model(**inputs)
        logits_cpu = out.logits.detach().float().cpu()
        total_loss += float(out.loss.item()) * labels_cpu.size(0)
        n += labels_cpu.size(0)
        all_pred += logits_cpu.argmax(-1).tolist()
        all_true += labels_cpu.tolist()
    return total_loss/n, f1_score(all_true, all_pred, average="macro"), all_true, all_pred

In [ ]:
# Train (the long-running cell — ~3-5 min on Colab T4, ~10 min on Mac MPS).
# Reuses df_train / df_val / df_test from §3 so the comparison is exact.
print(f"reusing splits from §3: train={len(df_train)} val={len(df_val)} test={len(df_test)}")

tok = AutoTokenizer.from_pretrained(cfg.model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    cfg.model_name, num_labels=len(EMOTION_LABELS),
    id2label=EMOTION_ID2LABEL, label2id=EMOTION_LABEL2ID,
    problem_type="single_label_classification",
).to(DEVICE)

ds_train = EmotionDS(df_train["text"].values, df_train["label_id"].values, tok, cfg.max_length)
ds_val   = EmotionDS(df_val["text"].values,   df_val["label_id"].values,   tok, cfg.max_length)
ds_test  = EmotionDS(df_test["text"].values,  df_test["label_id"].values,  tok, cfg.max_length)
train_loader = DataLoader(ds_train, batch_size=cfg.batch_size, shuffle=True)
val_loader   = DataLoader(ds_val,   batch_size=cfg.eval_bs)
test_loader  = DataLoader(ds_test,  batch_size=cfg.eval_bs)

no_decay = ["bias", "LayerNorm.weight"]
grouped = [
    {"params": [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     "weight_decay": cfg.weight_decay},
    {"params": [p for n,p in model.named_parameters() if     any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
]
optim = AdamW(grouped, lr=cfg.lr)
total_steps = cfg.epochs * len(train_loader)
sched = get_linear_schedule_with_warmup(optim, int(cfg.warmup_frac*total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

history = []; best_f1 = -1.0; best_epoch = -1; bad = 0
best_path = cfg.out_dir/"pytorch_model.bin"

for epoch in range(1, cfg.epochs+1):
    t0 = time.time(); model.train(); tloss, nb = 0.0, 0
    for batch in train_loader:
        batch = {k: v.to(DEVICE, non_blocking=True) for k,v in batch.items()}
        optim.zero_grad(set_to_none=True)
        if USE_AMP:
            with torch.amp.autocast("cuda", dtype=torch.float16):
                out = model(**batch)
            scaler.scale(out.loss).backward()
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optim); scaler.update()
        else:
            out = model(**batch); out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optim.step()
        sched.step(); tloss += out.loss.item(); nb += 1
    train_loss = tloss/nb
    val_loss, val_f1, _, _ = evaluate(model, val_loader, DEVICE)
    dt = time.time()-t0
    print(f"epoch {epoch}/{cfg.epochs}  train_loss={train_loss:.4f}  "
          f"val_loss={val_loss:.4f}  val_macro_f1={val_f1:.4f}  ({dt:.0f}s)")
    history.append(dict(epoch=epoch, train_loss=train_loss,
                        val_loss=val_loss, val_macro_f1=val_f1, seconds=dt))
    if val_f1 > best_f1 + 1e-6:
        best_f1, best_epoch, bad = val_f1, epoch, 0
        torch.save(model.state_dict(), best_path)
        print(f"  ↑ new best (val_macro_f1={val_f1:.4f}) — saved")
    else:
        bad += 1
        if bad >= cfg.patience:
            print(f"  ⨯ early stop after {bad} non-improving epochs"); break

print(f"\nbest val_macro_f1={best_f1:.4f} at epoch {best_epoch}")

In [ ]:
# Restore best, eval on test, save artefacts + plots.
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
test_loss, test_f1, y_true, y_pred = evaluate(model, test_loader, DEVICE)
test_acc = accuracy_score(y_true, y_pred)
print(f"=== TEST ===  loss={test_loss:.4f}  acc={test_acc:.4f}  macro_f1={test_f1:.4f}\n")
print(classification_report(y_true, y_pred, target_names=EMOTION_LABELS, digits=4))

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(EMOTION_LABELS))))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
h = pd.DataFrame(history)
axes[0].plot(h["epoch"], h["train_loss"], label="train")
axes[0].plot(h["epoch"], h["val_loss"], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(); axes[0].set_title("loss")
axes[1].plot(h["epoch"], h["val_macro_f1"], color="green", marker="o")
axes[1].axhline(best_f1, ls="--", color="grey", label=f"best={best_f1:.4f}")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("macro F1"); axes[1].legend()
axes[1].set_title("val macro F1"); plt.tight_layout()
plt.savefig(cfg.out_dir/"history.png", dpi=130, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(EMOTION_LABELS))); ax.set_yticks(range(len(EMOTION_LABELS)))
ax.set_xticklabels(EMOTION_LABELS, rotation=45, ha="right")
ax.set_yticklabels(EMOTION_LABELS)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title(f"BERTurk test confusion (acc={test_acc:.3f}  f1={test_f1:.3f})")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i,j]), ha="center", va="center",
                color="white" if cm[i,j] > cm.max()/2 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); plt.tight_layout()
plt.savefig(cfg.out_dir/"confusion_matrix.png", dpi=130, bbox_inches="tight"); plt.show()

tok.save_pretrained(cfg.out_dir/"tokenizer")
with open(cfg.out_dir/"test_metrics.json","w") as f:
    json.dump({"test_loss":test_loss,"test_acc":test_acc,
               "test_macro_f1":test_f1,"best_val_macro_f1":best_f1,
               "best_epoch":best_epoch,"labels":EMOTION_LABELS}, f, indent=2)
print(f"\nArtefacts → {cfg.out_dir}")

# ---- Head-to-head with the §3 baselines ----
print("\n" + "="*60)
print("BASELINE vs BERTURK (same test set, same split)")
print("="*60)
comp = pd.DataFrame([
    {"model": "TF-IDF + LogReg",     **baseline_results["logreg"]},
    {"model": "TF-IDF + Linear SVM", **baseline_results["svm"]},
    {"model": "BERTurk (fine-tuned)",
     "acc": float(test_acc), "macro_f1": float(test_f1),
     "per_class_f1": f1_score(y_true, y_pred, average=None).tolist()},
])
print(comp[["model", "acc", "macro_f1"]].round(4).to_string(index=False))
delta_lr  = test_f1 - baseline_results["logreg"]["macro_f1"]
delta_svm = test_f1 - baseline_results["svm"]["macro_f1"]
print(f"\nΔ macro F1 vs LogReg : {delta_lr:+.4f}")
print(f"Δ macro F1 vs SVM    : {delta_svm:+.4f}")
if max(delta_lr, delta_svm) < 0.01:
    print("\n→ BERTurk's advantage on this dataset is small. The corpus has "
          "highly discriminative emotion vocabulary (see §2 EDA top tokens) "
          "so TF-IDF already captures most of the signal — a useful "
          "reminder that Transformers don't automatically dominate.")
comp.to_csv(OUT/"model_comparison.csv", index=False)
print(f"\nSaved comparison → {OUT}/model_comparison.csv")

## §5 XLM-RoBERTa cross-lingual comparison (rubric §4)

A second Transformer for an apples-to-apples head-to-head. Two natural questions:
- Does Turkish-specific pretraining (BERTurk, ~110 M params, ~ 35 GB Turkish corpus) beat cross-lingual pretraining (XLM-R-base, ~278 M params, 100-language CC-100 corpus) on a Turkish task?
- How much of BERTurk's edge is the Turkish-specific corpus vs the smaller, easier-to-fine-tune size?

Same loop, same splits, same hyperparameters as §4 — the only thing that changes is `model_name`.

In [ ]:
# Train XLM-RoBERTa on the same splits + same training recipe as BERTurk.
XLMR_NAME = "xlm-roberta-base"
xlmr_out  = OUT / "xlmr_emotion"; xlmr_out.mkdir(parents=True, exist_ok=True)

print(f"Loading {XLMR_NAME}…")
tok_x   = AutoTokenizer.from_pretrained(XLMR_NAME)
model_x = AutoModelForSequenceClassification.from_pretrained(
    XLMR_NAME, num_labels=len(EMOTION_LABELS),
    id2label=EMOTION_ID2LABEL, label2id=EMOTION_LABEL2ID,
    problem_type="single_label_classification",
).to(DEVICE)
print(f"params: {sum(p.numel() for p in model_x.parameters())/1e6:.1f} M  "
      f"(vs BERTurk {sum(p.numel() for p in model.parameters())/1e6:.1f} M)")

# Datasets — same splits, new tokenizer
ds_train_x = EmotionDS(df_train["text"].values, df_train["label_id"].values, tok_x, cfg.max_length)
ds_val_x   = EmotionDS(df_val["text"].values,   df_val["label_id"].values,   tok_x, cfg.max_length)
ds_test_x  = EmotionDS(df_test["text"].values,  df_test["label_id"].values,  tok_x, cfg.max_length)
train_loader_x = DataLoader(ds_train_x, batch_size=cfg.batch_size, shuffle=True)
val_loader_x   = DataLoader(ds_val_x,   batch_size=cfg.eval_bs)
test_loader_x  = DataLoader(ds_test_x,  batch_size=cfg.eval_bs)

# Optimizer + scheduler — identical recipe to BERTurk
no_decay = ["bias", "LayerNorm.weight"]
grouped_x = [
    {"params": [p for n,p in model_x.named_parameters() if not any(nd in n for nd in no_decay)],
     "weight_decay": cfg.weight_decay},
    {"params": [p for n,p in model_x.named_parameters() if     any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
]
optim_x = AdamW(grouped_x, lr=cfg.lr)
total_steps_x = cfg.epochs * len(train_loader_x)
sched_x  = get_linear_schedule_with_warmup(
    optim_x, int(cfg.warmup_frac*total_steps_x), total_steps_x)
scaler_x = torch.amp.GradScaler("cuda", enabled=USE_AMP)

# Train
history_x   = []; best_f1_x = -1.0; best_epoch_x = -1; bad_x = 0
best_path_x = xlmr_out / "pytorch_model.bin"

for epoch in range(1, cfg.epochs+1):
    t0 = time.time(); model_x.train(); tloss, nb = 0.0, 0
    for batch in train_loader_x:
        batch = {k: v.to(DEVICE, non_blocking=True) for k,v in batch.items()}
        optim_x.zero_grad(set_to_none=True)
        if USE_AMP:
            with torch.amp.autocast("cuda", dtype=torch.float16):
                out = model_x(**batch)
            scaler_x.scale(out.loss).backward()
            scaler_x.unscale_(optim_x)
            torch.nn.utils.clip_grad_norm_(model_x.parameters(), cfg.grad_clip)
            scaler_x.step(optim_x); scaler_x.update()
        else:
            out = model_x(**batch); out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model_x.parameters(), cfg.grad_clip)
            optim_x.step()
        sched_x.step(); tloss += out.loss.item(); nb += 1
    train_loss = tloss/nb
    val_loss_x, val_f1_x, _, _ = evaluate(model_x, val_loader_x, DEVICE)
    dt = time.time() - t0
    print(f"epoch {epoch}/{cfg.epochs}  train_loss={train_loss:.4f}  "
          f"val_loss={val_loss_x:.4f}  val_macro_f1={val_f1_x:.4f}  ({dt:.0f}s)")
    history_x.append(dict(epoch=epoch, train_loss=train_loss,
                          val_loss=val_loss_x, val_macro_f1=val_f1_x))
    if val_f1_x > best_f1_x + 1e-6:
        best_f1_x, best_epoch_x, bad_x = val_f1_x, epoch, 0
        torch.save(model_x.state_dict(), best_path_x)
        print(f"  ↑ new best — saved")
    else:
        bad_x += 1
        if bad_x >= cfg.patience:
            print(f"  ⨯ early stop"); break

print(f"\nbest XLM-R val_macro_f1={best_f1_x:.4f} at epoch {best_epoch_x}")

In [ ]:
# Restore best XLM-R, eval on test, plot, save, then 4-way comparison.
model_x.load_state_dict(torch.load(best_path_x, map_location=DEVICE))
test_loss_x, test_f1_x, y_true_x, y_pred_x = evaluate(model_x, test_loader_x, DEVICE)
test_acc_x = accuracy_score(y_true_x, y_pred_x)
print(f"=== XLM-R TEST ===  acc={test_acc_x:.4f}  macro_f1={test_f1_x:.4f}\n")
print(classification_report(y_true_x, y_pred_x,
      target_names=EMOTION_LABELS, digits=4))

cm_x = confusion_matrix(y_true_x, y_pred_x,
                        labels=list(range(len(EMOTION_LABELS))))
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_x, cmap="Blues")
ax.set_xticks(range(len(EMOTION_LABELS))); ax.set_yticks(range(len(EMOTION_LABELS)))
ax.set_xticklabels(EMOTION_LABELS, rotation=45, ha="right")
ax.set_yticklabels(EMOTION_LABELS)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title(f"XLM-R test confusion (acc={test_acc_x:.3f}  f1={test_f1_x:.3f})")
for i in range(cm_x.shape[0]):
    for j in range(cm_x.shape[1]):
        ax.text(j, i, str(cm_x[i,j]), ha="center", va="center",
                color="white" if cm_x[i,j] > cm_x.max()/2 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); plt.tight_layout()
plt.savefig(xlmr_out/"confusion_matrix.png", dpi=130, bbox_inches="tight"); plt.show()

with open(xlmr_out/"test_metrics.json","w") as f:
    json.dump({"test_acc":test_acc_x,"test_macro_f1":test_f1_x,
               "best_val_macro_f1":best_f1_x,"best_epoch":best_epoch_x,
               "labels":EMOTION_LABELS}, f, indent=2)

# ---- 4-way head-to-head comparison ----
print("\n" + "="*60)
print("4-WAY COMPARISON (same test set, same split)")
print("="*60)
comp = pd.DataFrame([
    {"model": "TF-IDF + LogReg",     **baseline_results["logreg"]},
    {"model": "TF-IDF + Linear SVM", **baseline_results["svm"]},
    {"model": "BERTurk (fine-tuned)",
     "acc": float(test_acc), "macro_f1": float(test_f1),
     "per_class_f1": f1_score(y_true,   y_pred,   average=None).tolist()},
    {"model": "XLM-R-base (fine-tuned)",
     "acc": float(test_acc_x), "macro_f1": float(test_f1_x),
     "per_class_f1": f1_score(y_true_x, y_pred_x, average=None).tolist()},
])
print(comp[["model","acc","macro_f1"]].round(4).to_string(index=False))

delta = test_f1_x - test_f1
print(f"\nΔ macro F1  XLM-R vs BERTurk : {delta:+.4f}")
if delta < -0.01:
    print("→ Turkish-specific pretraining (BERTurk) wins on this Turkish task.")
elif delta > 0.01:
    print("→ XLM-R surprisingly matches/beats BERTurk despite no Turkish-specific "
          "pretraining — likely due to its 2.5× larger parameter budget.")
else:
    print("→ Within noise — both Transformers solve this task essentially perfectly. "
          "On this dataset, model choice barely matters once you pick a Transformer.")

comp.to_csv(OUT/"model_comparison.csv", index=False)

# Plot side-by-side macro F1 bars
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(comp["model"], comp["macro_f1"],
              color=["#A0A0A0","#A0A0A0","#4C78A8","#54A24B"])
ax.set_ylabel("test macro F1")
ax.set_title("Emotion classification — 4-way comparison")
ax.set_ylim(0.80, 1.01)
for b, v in zip(bars, comp["macro_f1"]):
    ax.text(b.get_x()+b.get_width()/2, v+0.003,
            f"{v:.3f}", ha="center", fontsize=10)
plt.xticks(rotation=15, ha="right"); plt.tight_layout()
plt.savefig(OUT/"model_comparison.png", dpi=130, bbox_inches="tight"); plt.show()

## §6 Synthetic user bundles

We don't have real per-author data, so we fabricate 25 users from the tweet pool: 5 *balanced* (uniform across emotions, control) + 20 *skewed* (4 per emotion × 5 emotions, 8/12 dominant). Skew ratio = 0.70 rounded to 8/12 = 0.67.

In [ ]:
N_BALANCED      = 5
N_PER_EMOTION   = 4
TWEETS_PER_USER = 12
SKEW_RATIO      = 0.70


def sample_user_tweets(df, rng, k, dominant, skew):
    if dominant is None:
        idx = rng.sample(range(len(df)), k)
    else:
        n_dom = round(k * skew); n_oth = k - n_dom
        dom_pool = df.index[df["label"] == dominant].tolist()
        oth_pool = df.index[df["label"] != dominant].tolist()
        idx = rng.sample(dom_pool, n_dom) + rng.sample(oth_pool, n_oth)
        rng.shuffle(idx)
    return [{"text": df.at[i, "text"],
             "true_emotion": df.at[i, "label"]} for i in idx]


def build_users(df):
    rng = random.Random(SEED); users = []; uid = 0
    for _ in range(N_BALANCED):
        users.append({"user_id": f"u{uid:02d}", "type": "balanced",
                      "dominant_emotion": None,
                      "tweets": sample_user_tweets(df, rng, TWEETS_PER_USER, None, SKEW_RATIO)})
        uid += 1
    for emo in EMOTION_LABELS:
        for _ in range(N_PER_EMOTION):
            users.append({"user_id": f"u{uid:02d}", "type": "skewed",
                          "dominant_emotion": emo,
                          "tweets": sample_user_tweets(df, rng, TWEETS_PER_USER, emo, SKEW_RATIO)})
            uid += 1
    return users


users = build_users(df_emo)
print(f"users: {len(users)}  (balanced={sum(1 for u in users if u['type']=='balanced')}, "
      f"skewed={sum(1 for u in users if u['type']=='skewed')})")
print("\nfirst 3 users — true label mix:")
for u in users[:3]:
    c = Counter(t["true_emotion"] for t in u["tweets"])
    mix = ", ".join(f"{e}:{c.get(e,0)}" for e in EMOTION_LABELS)
    print(f"  {u['user_id']}  type={u['type']:8s}  dom={u['dominant_emotion'] or '—':8s}  {mix}")

with open(OUT/"users.json","w",encoding="utf-8") as f:
    json.dump(users, f, ensure_ascii=False, indent=2)

## §7 User vectors (Step 5)

Run BERTurk over each user's bundle, aggregate into a per-user affect signal (mean softmax over 12 tweets) + topical signal (top-K TF-IDF keywords fit on the *full* corpus so common emotion words don't dominate every user).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

TOP_K_KEYWORDS = 6


class TextDS(Dataset):
    def __init__(self, texts, tok, max_length):
        self.texts = list(texts); self.tok = tok; self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True,
                       max_length=self.max_length,
                       padding="max_length", return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0)}


@torch.no_grad()
def predict_probs(model, tok, texts, device, batch_size=32, max_length=64):
    ds = TextDS(texts, tok, max_length)
    loader = DataLoader(ds, batch_size=batch_size)
    model.eval(); probs = []
    for batch in loader:
        batch = {k: v.to(device, non_blocking=True) for k,v in batch.items()}
        out = model(**batch)
        logits = out.logits.detach().float().cpu()
        probs.append(torch.softmax(logits, dim=-1).numpy())
    return np.concatenate(probs, axis=0)


# Fit TF-IDF on full corpus
tfidf = TfidfVectorizer(
    lowercase=True, token_pattern=r"[a-zçğıöşüâîû]{3,}",
    stop_words=list(TR_STOPWORDS), ngram_range=(1,1),
    min_df=2, max_features=20000,
)
tfidf.fit(df_emo["text"].tolist())
print(f"TF-IDF vocab: {len(tfidf.get_feature_names_out()):,}")

# Predict over all user tweets in one pass
all_texts = [t["text"] for u in users for t in u["tweets"]]
probs = predict_probs(model, tok, all_texts, DEVICE)

cursor = 0; user_vectors = []
for u in users:
    n = len(u["tweets"])
    p = probs[cursor:cursor+n]; cursor += n
    mean_probs = p.mean(axis=0); argmax_pred = p.argmax(axis=1)
    am_counts = Counter(EMOTION_LABELS[i] for i in argmax_pred.tolist())

    joined = " ".join(t["text"] for t in u["tweets"])
    scores = tfidf.transform([joined]).toarray()[0]
    vocab = tfidf.get_feature_names_out()
    top_idx = np.argsort(-scores)[:TOP_K_KEYWORDS]
    keywords = [vocab[i] for i in top_idx if scores[i] > 0]

    tweets_pred = [{"text": t["text"], "true": t["true_emotion"],
                    "pred": EMOTION_LABELS[int(am)],
                    "conf": float(pr[int(am)])}
                   for t, pr, am in zip(u["tweets"], p, argmax_pred)]

    user_vectors.append({
        "user_id": u["user_id"], "type": u["type"],
        "true_dominant": u["dominant_emotion"],
        "predicted_dominant": EMOTION_LABELS[int(mean_probs.argmax())],
        "dominant_confidence": float(mean_probs.max()),
        "n_tweets": n,
        "emotion_dist_mean": {EMOTION_LABELS[i]: float(mean_probs[i])
                              for i in range(len(EMOTION_LABELS))},
        "emotion_dist_argmax_counts": {e: int(am_counts.get(e, 0)) for e in EMOTION_LABELS},
        "top_keywords": keywords,
        "tweets_predicted": tweets_pred,
    })

# Eval: predicted-vs-true dominant for skewed users
skewed = [u for u in user_vectors if u["true_dominant"] is not None]
hits = sum(u["true_dominant"] == u["predicted_dominant"] for u in skewed)
print(f"\nSkewed users: {hits}/{len(skewed)} dominant-match "
      f"({100*hits/len(skewed):.1f}%)")
print("balanced users (no ground truth):")
for u in user_vectors:
    if u["true_dominant"] is None:
        print(f"  {u['user_id']}  pred={u['predicted_dominant']:8s}  "
              f"conf={u['dominant_confidence']:.2f}")

with open(OUT/"user_vectors.json","w",encoding="utf-8") as f:
    json.dump(user_vectors, f, ensure_ascii=False, indent=2)

## §8 Turkish persona templates (Step 6)

Three variants per user for the Step-9 ablation:
- **full**: real affect dist + keywords + 3 example tweets
- **random**: same template structure but a *donor* user's affect/keywords/tweets — controls for "longer prompt → better ad"
- **generic**: one neutral sentence — controls for "any instruction → ad"

In [ ]:
N_EXAMPLE_TWEETS = 3
LOW_CONF_THRESH  = 0.45  # below this we say "no clear emotion"

EMO_TR = {"angry":"öfkeli","fear":"korkulu","happy":"mutlu",
          "surprise":"şaşkın","sad":"üzgün"}


def _pct(emotion_dist):
    items = sorted(emotion_dist.items(), key=lambda kv: -kv[1])
    return ", ".join(f"%{round(100*v):d} {EMO_TR[k]}" for k,v in items)


def _pick_examples(uv, rng):
    tweets = uv["tweets_predicted"]; dom = uv["predicted_dominant"]
    conf = uv["dominant_confidence"]
    if conf >= LOW_CONF_THRESH:
        dom_t = sorted([t for t in tweets if t["pred"] == dom],
                       key=lambda t: -t["conf"])
        oth_t = [t for t in tweets if t["pred"] != dom]
        chosen = dom_t[:2] + ([rng.choice(oth_t)] if oth_t else dom_t[2:3])
    else:
        chosen = rng.sample(tweets, min(N_EXAMPLE_TWEETS, len(tweets)))
    return [c["text"] for c in chosen][:N_EXAMPLE_TWEETS]


def render_full(uv, rng):
    conf = uv["dominant_confidence"]; dom = uv["predicted_dominant"]
    kws = uv.get("top_keywords", []); examples = _pick_examples(uv, rng)
    if conf >= LOW_CONF_THRESH:
        affect = (f"Bu Türkçe Twitter kullanıcısı son paylaşımlarında ağırlıklı "
                  f"olarak {EMO_TR[dom]} bir duygu durumu sergilemektedir.")
    else:
        affect = ("Bu Türkçe Twitter kullanıcısında belirgin bir duygu eğilimi "
                  "gözlenmemekte; farklı duygular arasında dengeli bir dağılım vardır.")
    dist  = "Duygu dağılımı: " + _pct(uv["emotion_dist_mean"]) + "."
    kw    = "Sıkça geçen içerik terimleri: " + ", ".join(kws) + "." if kws else ""
    ex    = "Örnek paylaşımları:\n" + "\n".join(f'  - "{t}"' for t in examples)
    return "\n".join(p for p in [affect, dist, kw, ex] if p)


def render_random(uv, donor_uv, rng):
    merged = {**uv,
        "predicted_dominant":  donor_uv["predicted_dominant"],
        "dominant_confidence": donor_uv["dominant_confidence"],
        "emotion_dist_mean":   donor_uv["emotion_dist_mean"],
        "top_keywords":        donor_uv.get("top_keywords", []),
        "tweets_predicted":    donor_uv["tweets_predicted"]}
    return render_full(merged, rng)


def render_generic(_uv, _rng):
    return ("Bu Türkçe konuşan bir sosyal medya kullanıcısıdır. "
            "Kişisel tercih, duygu durumu veya özel ilgi alanları hakkında "
            "ek bilgi bulunmamaktadır.")


def build_personas(uvs, rng):
    donors = []
    for i, u in enumerate(uvs):
        cands = [v for j,v in enumerate(uvs)
                 if j != i and v["predicted_dominant"] != u["predicted_dominant"]]
        if not cands:
            cands = [v for j,v in enumerate(uvs) if j != i]
        donors.append(rng.choice(cands))
    return [{
        "user_id": u["user_id"], "type": u["type"],
        "true_dominant": u["true_dominant"],
        "predicted_dominant":  u["predicted_dominant"],
        "dominant_confidence": u["dominant_confidence"],
        "personas": {
            "full":    render_full(u, rng),
            "random":  render_random(u, donor, rng),
            "generic": render_generic(u, rng),
        },
        "random_donor_id":      donor["user_id"],
        "random_donor_emotion": donor["predicted_dominant"],
    } for u, donor in zip(uvs, donors)]


personas = build_personas(user_vectors, random.Random(SEED))
with open(OUT/"personas.json","w",encoding="utf-8") as f:
    json.dump(personas, f, ensure_ascii=False, indent=2)

# Show one skewed + one balanced
skewed = next(p for p in personas
              if p["type"]=="skewed" and p["dominant_confidence"]>=LOW_CONF_THRESH)
bal = next(p for p in personas if p["type"]=="balanced")
for tag, p in [("SKEWED", skewed), ("BALANCED", bal)]:
    print("="*60); print(f"{tag}: {p['user_id']}  pred={p['predicted_dominant']}  conf={p['dominant_confidence']:.2f}")
    for v in ("full","random","generic"):
        print(f"\n--- {v} ---\n{p['personas'][v]}")
    print()

## §9 Product catalog (Step 7)

10 fictional Turkish products, deliberately spanning categories where emotion-aware pitching has different natural angles (e.g., a mindfulness app sells differently to an angry user vs a fearful user).

In [ ]:
PRODUCTS = [
    {
        "product_id": "p01",
        "name": "LezzetKapım",
        "category": "yemek_teslimat",
        "category_en": "comfort food delivery",
        "description": "Restoran kalitesindeki ev yemeklerini 30 dakika içinde kapınıza getiren bir teslimat hizmeti.",
        "key_features": [
            "30 dk teslimat",
            "ev yapımı menü",
            "her gün değişen seçenekler"
        ],
        "price_hint": "öğün başı 120-180 TL"
    },
    {
        "product_id": "p02",
        "name": "Sakinlik+",
        "category": "wellness_app",
        "category_en": "mindfulness app",
        "description": "Türkçe nefes egzersizleri, kısa meditasyonlar ve günlük duygu takibi sunan bir mobil uygulama.",
        "key_features": [
            "5 dk meditasyonlar",
            "uyku sesleri",
            "günlük duygu günlüğü"
        ],
        "price_hint": "ücretsiz / Premium 49 TL/ay"
    },
    {
        "product_id": "p03",
        "name": "MaceraBileti",
        "category": "seyahat",
        "category_en": "adventure travel package",
        "description": "Türkiye içi sürpriz hafta sonu kaçamakları planlayan bir seyahat paketi servisi.",
        "key_features": [
            "sürpriz destinasyon",
            "ulaşım+konaklama dahil",
            "küçük gruplarla"
        ],
        "price_hint": "kişi başı 2.500-4.000 TL"
    },
    {
        "product_id": "p04",
        "name": "BütçeRehberi",
        "category": "fintech",
        "category_en": "personal finance app",
        "description": "Harcamalarınızı otomatik kategorize eden, aylık bütçe hedefleri ve kontrollü tasarruf önerileri sunan bir kişisel finans uygulaması.",
        "key_features": [
            "otomatik harcama analizi",
            "ay sonu bütçe uyarıları",
            "akıllı tasarruf önerileri"
        ],
        "price_hint": "ücretsiz / Pro 39 TL/ay"
    },
    {
        "product_id": "p05",
        "name": "FormaSet",
        "category": "fitness",
        "category_en": "fitness program",
        "description": "Evde yapılabilen 4 haftalık antrenman programları ve Türkçe video rehberlerinden oluşan dijital bir fitness paketi.",
        "key_features": [
            "evde 20 dk antrenman",
            "yeni başlayan-ileri seviye",
            "ilerleme takibi"
        ],
        "price_hint": "tek seferlik 299 TL"
    },
    {
        "product_id": "p06",
        "name": "GeceMarşı",
        "category": "yiyecek_icecek",
        "category_en": "wellness drink",
        "description": "Doğal bitki özleriyle hazırlanmış, gece yatmadan önce içilen rahatlatıcı bir bitki çayı serisi.",
        "key_features": [
            "uyku öncesi içecek",
            "papatya & melisa karışımı",
            "kafeinsiz"
        ],
        "price_hint": "20'li paket 89 TL"
    },
    {
        "product_id": "p07",
        "name": "DiziKulesi",
        "category": "eglence",
        "category_en": "streaming service",
        "description": "Türkçe ve dünya dizilerini reklamsız sunan bir video akış platformu.",
        "key_features": [
            "Türkçe yapımlar",
            "reklamsız izleme",
            "haftalık yeni bölümler"
        ],
        "price_hint": "aylık 79 TL"
    },
    {
        "product_id": "p08",
        "name": "Yansı",
        "category": "kozmetik",
        "category_en": "skincare line",
        "description": "Hassas ciltler için tasarlanmış nemlendirici, temizleyici ve serumdan oluşan üçlü cilt bakım seti.",
        "key_features": [
            "hassas cilt formülü",
            "doğal içerikler",
            "dermatolog onaylı"
        ],
        "price_hint": "set 349 TL"
    },
    {
        "product_id": "p09",
        "name": "ÖğrenÇağı",
        "category": "egitim",
        "category_en": "online course platform",
        "description": "Veri bilimi, tasarım ve dil eğitimleri sunan, Türkçe alt yazılı uzaktan eğitim platformu.",
        "key_features": [
            "uzman eğitmenler",
            "sertifikalı kurslar",
            "kendi temponda öğrenme"
        ],
        "price_hint": "kurs başı 199-499 TL"
    },
    {
        "product_id": "p10",
        "name": "EvGözcü",
        "category": "akilli_ev",
        "category_en": "smart home device",
        "description": "Telefondan canlı izleme, hareket bildirimi ve gece görüşü özellikli bir akıllı ev güvenlik kamerası.",
        "key_features": [
            "1080p gece görüşü",
            "anlık hareket uyarısı",
            "kolay kurulum"
        ],
        "price_hint": "tek kamera 899 TL"
    }
]

with open(OUT/"products.json","w",encoding="utf-8") as f:
    json.dump(PRODUCTS, f, ensure_ascii=False, indent=2)

print(f"{'id':<5} {'name':<14} {'category_en':<22} {'price'}")
print("-"*70)
for p in PRODUCTS:
    print(f"{p['product_id']:<5} {p['name']:<14} {p['category_en']:<22} {p['price_hint']}")

## §10 LLM ad generator (Step 8)

This notebook uses **Groq** (Llama-3.3-70B-Versatile) as the only LLM backend. A deterministic stub is included as the no-key fallback so the notebook still runs end-to-end without an API key (useful for graders or for verifying the pipeline plumbing without burning calls).

**Get a free Groq key in 30 seconds:**
1. Visit [console.groq.com/keys](https://console.groq.com/keys) → sign in → **Create API Key**
2. In Colab: click the **🔑 Secrets** icon in the left sidebar → **Add new secret**
3. Name it `GROQ_API_KEY`, paste the key, toggle **Notebook access** ON
4. Re-run the next cell — it will detect the key automatically

Disk cache keyed by `SHA-256(system + user + backend:model)` — re-runs are free.

In [ ]:
# --- Groq API key detection ---
# Checks (1) Colab Secrets, then (2) shell env. Falls back to stub if not set.
GROQ_KEY_SET = False

def _try_load_secret(name: str) -> bool:
    if os.environ.get(name):
        return True
    try:
        from google.colab import userdata  # type: ignore
        try:
            v = userdata.get(name)
            if v:
                os.environ[name] = v
                return True
        except Exception:
            pass
    except ImportError:
        pass
    return False

if _try_load_secret("GROQ_API_KEY"):
    GROQ_KEY_SET = True
    print("✓ GROQ_API_KEY loaded")
else:
    print("⚠️  No GROQ_API_KEY found — will fall back to BACKEND='stub'.")
    print("    To enable real Llama-3.3-70B ad generation:")
    print("      1. Get a free key (no card):  https://console.groq.com/keys")
    print("      2. Colab → 🔑 Secrets (left sidebar) → Add secret")
    print("         name='GROQ_API_KEY', paste, toggle Notebook access ON")
    print("      3. Re-run THIS cell.")

In [ ]:
# Backend selection: groq if key is set, else stub fallback.
BACKEND = "groq" if GROQ_KEY_SET else "stub"
MODEL   = None  # None = backend-specific default (llama-3.3-70b-versatile)
print(f"BACKEND = {BACKEND!r}")
CACHE_DIR = OUT/"ads_cache"; CACHE_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = (
    "Sen kısa ve etkili Türkçe reklam metinleri yazan bir kreatif yazarsın. "
    "Kuralların:\n"
    "1. Reklam 40-70 kelime arasında ve TEK paragraf olmalı.\n"
    "2. Kullanıcının ruh halini ya da kişiliğini AÇIKÇA söyleme — buna göre "
    "uygun bir TON seç ama 'sen şu duygudasın' gibi ifadeler kullanma.\n"
    "3. Türkçe akıcı ve doğal olsun, çeviri kokmasın.\n"
    "4. Ürünün adını ve en az bir özelliğini metne yedir.\n"
    "5. Sonuç sadece reklam metnidir; başlık, açıklama veya emoji ekleme."
)


def build_user_prompt(persona_text, product):
    feats = ", ".join(product.get("key_features", []))
    return ("Aşağıdaki kullanıcı profili ve ürüne uygun bir reklam metni yaz.\n\n"
            f"KULLANICI PROFİLİ:\n{persona_text}\n\n"
            f"ÜRÜN:\nİsim: {product['name']}\n"
            f"Açıklama: {product['description']}\n"
            f"Öne çıkan özellikler: {feats}\n"
            f"Fiyat: {product.get('price_hint','—')}\n\n"
            "REKLAM METNİ:")


def _cache_key(system, user, model):
    h = hashlib.sha256()
    h.update(model.encode()); h.update(b"\n---\n")
    h.update(system.encode()); h.update(b"\n---\n"); h.update(user.encode())
    return h.hexdigest()[:24]


def _cache_get(key):
    f = CACHE_DIR/f"{key}.json"
    if not f.exists(): return None
    with open(f, encoding="utf-8") as fh: return json.load(fh)


def _cache_put(key, payload):
    with open(CACHE_DIR/f"{key}.json", "w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=2)


def _gen_stub(system, user, model):
    hints = {"öfkeli":"kontrolü geri al","korkulu":"huzurlu bir nefes",
             "üzgün":"kendine bir mola ver","şaşkın":"yeni bir keşif",
             "mutlu":"iyi anı sürdür"}
    earliest = (10**9, "doğal")
    for emo, hint in hints.items():
        idx = user.find(emo)
        if idx != -1 and idx < earliest[0]: earliest = (idx, hint)
    tone = earliest[1]; name = "ürün"; feature = ""
    for line in user.splitlines():
        if line.startswith("İsim:"):       name = line.split(":",1)[1].strip()
        if line.startswith("Öne çıkan"):   feature = line.split(":",1)[1].split(",")[0].strip()
    return (f"{tone.capitalize()} için {name}. "
            f"{feature.capitalize()} ile gününe küçük bir fark kat. "
            f"Hemen dene, kendine iyi gel.")


# Groq free tier on llama-3.3-70b-versatile is ~30 req/min. Pace at ~2.5s.
_GROQ_MIN_INTERVAL = 2.5

def _gen_groq(system, user, model):
    from groq import Groq
    client = Groq(api_key=os.environ["GROQ_API_KEY"])

    last = getattr(_gen_groq, "_last_call", 0.0)
    elapsed = time.time() - last
    if elapsed < _GROQ_MIN_INTERVAL:
        time.sleep(_GROQ_MIN_INTERVAL - elapsed)

    def _call():
        return client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": system},
                      {"role": "user",   "content": user}],
            temperature=0.7,
            max_tokens=600,
        )
    try:
        resp = _call()
    except Exception as e:
        msg = str(e).lower()
        if "429" in msg or "rate" in msg or "quota" in msg:
            print("  groq rate-limited, sleeping 30s and retrying once…")
            time.sleep(30)
            resp = _call()
        else:
            raise
    _gen_groq._last_call = time.time()
    return resp.choices[0].message.content.strip()


_BACKENDS = {"stub": _gen_stub, "groq": _gen_groq}


def generate_ad(persona_text, product, backend="groq", model=None, use_cache=True):
    if model is None:
        model = {"stub": "stub-v1",
                 "groq": "llama-3.3-70b-versatile"}[backend]
    system = SYSTEM_PROMPT; user = build_user_prompt(persona_text, product)
    key = _cache_key(system, user, f"{backend}:{model}")
    if use_cache and (hit := _cache_get(key)):
        return hit["ad_text"], {**hit["meta"], "cache_hit": True}
    t0 = time.time()
    ad_text = _BACKENDS[backend](system, user, model)
    meta = {"backend":backend,"model":model,"cache_hit":False,
            "latency_seconds":round(time.time()-t0,3),"key":key}
    _cache_put(key, {"ad_text":ad_text,"meta":meta,"system":system,"user":user})
    return ad_text, meta


# Demo: u05 (angry) × p02 (Sakinlik+) under all 3 variants
demo_user = next(p for p in personas if p["user_id"]=="u05")
demo_prod = next(p for p in PRODUCTS if p["product_id"]=="p02")
print(f"=== DEMO: {demo_user['user_id']}/{demo_user['true_dominant']} × {demo_prod['name']} (backend={BACKEND}) ===\n")
for variant in ("full","random","generic"):
    ad, meta = generate_ad(demo_user["personas"][variant], demo_prod,
                           backend=BACKEND, model=MODEL)
    print(f"--- {variant} ---\n{ad}\n[cache_hit={meta['cache_hit']}, latency={meta['latency_seconds']}s]\n")

## §11 Sweep (Step 9)

5 emotions × 2 products (one natural-fit + one adjacent) × 3 variants = **30 ads**. Adjacent products force the LLM to actually personalize, not just match obvious categories.

In [ ]:
PAIR_PLAN = [
    ("angry",    "u05", [("p02","natural"),("p05","adjacent")]),
    ("fear",     "u09", [("p06","natural"),("p10","adjacent")]),
    ("happy",    "u13", [("p03","natural"),("p08","adjacent")]),
    ("surprise", "u17", [("p09","natural"),("p02","adjacent")]),
    ("sad",      "u21", [("p01","natural"),("p07","adjacent")]),
]
VARIANTS = ["full","random","generic"]
personas_by_id = {p["user_id"]: p for p in personas}
products_by_id = {p["product_id"]: p for p in PRODUCTS}

rows = []; cache_hits = 0; total_latency = 0.0
print(f"[backend={BACKEND}] generating {sum(len(ps) for _,_,ps in PAIR_PLAN) * len(VARIANTS)} ads")
for emotion, uid, prod_list in PAIR_PLAN:
    upers = personas_by_id[uid]
    for pid, fit in prod_list:
        prod = products_by_id[pid]
        for variant in VARIANTS:
            ad, meta = generate_ad(upers["personas"][variant], prod,
                                   backend=BACKEND, model=MODEL)
            rows.append({"emotion":emotion,"user_id":uid,
                "true_dominant":upers["true_dominant"],
                "predicted_dominant":upers["predicted_dominant"],
                "product_id":pid,"product_name":prod["name"],
                "category_en":prod["category_en"],"fit":fit,
                "variant":variant,"backend":meta["backend"],"model":meta["model"],
                "cache_hit":meta["cache_hit"],
                "latency_seconds":meta["latency_seconds"],"ad_text":ad})
            cache_hits += int(meta["cache_hit"])
            total_latency += meta["latency_seconds"]

with open(OUT/"ads_results.json","w",encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
pd.DataFrame(rows).to_csv(OUT/"ads_results.csv", index=False)
print(f"\nGenerated {len(rows)} ads ({cache_hits} cache hits, "
      f"{len(rows)-cache_hits} live calls, total {total_latency:.1f}s)")
print(f"\n--- preview (first 3) ---")
for r in rows[:3]:
    print(f"\n[{r['emotion']}/{r['user_id']} × {r['product_name']} / {r['variant']}]")
    print(r["ad_text"])

## §12 Evaluation (Step 10)

**Layer 2** — programmatic metrics (no LLM): compliance, personalization, variant differentiation.
**Layer 3** — LLM-as-judge with anonymised A/B/C reveal map. The judge uses whatever `BACKEND` is set in §10 (default: Groq). With no API key the stub judge derives 1-5 scores from Layer-2 features so the pipeline still runs end-to-end.

In [ ]:
WORD_RE  = re.compile(r"[a-zçğıöşüâîû]+", flags=re.IGNORECASE)
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\U0001F600-\U0001F64F☀-➿]", flags=re.UNICODE)
CALLOUT_PATTERNS = [
    r"\b(siz|sen)\s+\w*(öfkeli|sinirli|üzgün|mutlu|korkulu|şaşkın|kaygılı)",
    r"\b(öfkeli|sinirli|üzgün|mutlu|korkulu|şaşkın|kaygılı)\s+(görünüyorsunuz|hissediyorsun|olmalısın|olduğun)",
    r"\bduygu\s+durumu(nuz|n)\b", r"\bruh\s+haliniz\b",
]
CALLOUT_RE = re.compile("|".join(CALLOUT_PATTERNS), flags=re.IGNORECASE)

EMOTION_AD_LEXICON = {
    "angry":   {"öfke","sinir","kontrol","rahatla","rahatlat","boşalt","patla","stres","gerginlik","sakin","tepki","soğukkanlı"},
    "fear":    {"korku","huzur","güven","güvenli","sakin","endişe","koru","korur","nefes","rahat","emin","garanti"},
    "happy":   {"mutlu","neşe","keyif","kutla","harika","eğlence","sürdür","iyi","güzel","parla","ışılda","an","anı"},
    "surprise":{"sürpriz","şaşkın","keşfet","yeni","merak","heyecan","beklenmedik","fark","büyülen"},
    "sad":     {"moral","mola","kendine","destek","umut","yalnız","şefkat","sıcak","ısı","rahatla","huzur","iyileş","yenile"},
}

def _tokenize(text): return [t.lower() for t in WORD_RE.findall(text)]
def _jaccard(a,b):
    sa, sb = set(a), set(b)
    return 1.0 if not sa and not sb else len(sa & sb)/len(sa | sb)


def evaluate_one(row, products_by_id, user_kw):
    ad = row["ad_text"]; toks = _tokenize(ad); n = len(toks); s = set(toks)
    prod = products_by_id[row["product_id"]]
    name_ok = prod["name"].lower() in ad.lower()
    feat_ok = any(any(w in s for w in _tokenize(f)) for f in prod.get("key_features",[]))
    callout = bool(CALLOUT_RE.search(ad)); emoji = bool(EMOJI_RE.search(ad))
    emo = row["true_dominant"] or row["predicted_dominant"]; emo_hits = 0
    if emo and emo in EMOTION_AD_LEXICON:
        lex = EMOTION_AD_LEXICON[emo]
        long_stems  = {w for w in lex if len(w)>=4}
        short_exact = {w for w in lex if len(w)<4}
        for t in toks:
            if t in short_exact: emo_hits += 1
            elif any(t.startswith(s2) for s2 in long_stems): emo_hits += 1
    kws = user_kw.get(row["user_id"], [])
    kw_overlap = sum(1 for kw in kws if kw.lower() in s)
    return {**{k: row[k] for k in
        ("user_id","true_dominant","predicted_dominant","product_id",
         "product_name","category_en","fit","variant","backend","model")},
        "word_count": n, "in_word_band": 40 <= n <= 70,
        "product_name_mentioned": bool(name_ok),
        "feature_mentioned": bool(feat_ok),
        "explicit_emotion_callout": callout, "has_emoji": emoji,
        "emotion_word_density": round(emo_hits / max(n,1), 4),
        "persona_keyword_overlap": kw_overlap, "ad_text": ad}


user_kw = {u["user_id"]: u.get("top_keywords", []) for u in user_vectors}
per_ad = [evaluate_one(r, products_by_id, user_kw) for r in rows]
with open(OUT/"eval_results.json","w",encoding="utf-8") as f:
    json.dump(per_ad, f, ensure_ascii=False, indent=2)

df_eval = pd.DataFrame(per_ad)
agg = df_eval.groupby("variant").agg(
    n=("ad_text","count"),
    mean_word_count=("word_count","mean"),
    pct_in_word_band=("in_word_band","mean"),
    pct_product_named=("product_name_mentioned","mean"),
    pct_feature_named=("feature_mentioned","mean"),
    pct_explicit_callout=("explicit_emotion_callout","mean"),
    pct_emoji=("has_emoji","mean"),
    mean_emotion_density=("emotion_word_density","mean"),
    mean_keyword_overlap=("persona_keyword_overlap","mean"),
).round(3)
agg.to_csv(OUT/"eval_summary.csv")
print("=== Per-variant aggregates ==="); print(agg.to_string()); print()

print("=== emotion-word density per (true_dominant, variant) ===")
pivot = df_eval.pivot_table(index="true_dominant", columns="variant",
    values="emotion_word_density", aggfunc="mean").round(3)
pivot["full−generic"] = (pivot["full"] - pivot["generic"]).round(3)
pivot["full−random"]  = (pivot["full"] - pivot["random"]).round(3)
print(pivot.to_string()); print()

diff_rows = []
for (uid, pid), g in df_eval.groupby(["user_id","product_id"]):
    cells = {r["variant"]: _tokenize(r["ad_text"]) for _, r in g.iterrows()}
    if not all(v in cells for v in ("full","random","generic")): continue
    diff_rows.append({"user_id":uid,"product_id":pid,
        "true_dominant": g["true_dominant"].iloc[0],
        "jaccard_full_vs_generic": round(_jaccard(cells["full"], cells["generic"]), 3),
        "jaccard_full_vs_random":  round(_jaccard(cells["full"], cells["random"]),  3)})
diff_df = pd.DataFrame(diff_rows)
diff_df.to_csv(OUT/"eval_diff_pairs.csv", index=False)
print("=== Variant differentiation (Jaccard, lower = more different) ===")
j_fg = diff_df["jaccard_full_vs_generic"]; j_fr = diff_df["jaccard_full_vs_random"]
print(f"  full vs generic   mean={j_fg.mean():.3f}  %cells with j<0.95: {100*(j_fg<0.95).mean():.0f}%")
print(f"  full vs random    mean={j_fr.mean():.3f}  %cells with j<0.95: {100*(j_fr<0.95).mean():.0f}%")

In [ ]:
# Layer 3 — LLM-as-judge with anonymised A/B/C reveal.
JUDGE_SYSTEM = (
    "Sen tarafsız bir Türkçe reklam değerlendirme uzmanısın. "
    "Sana bir kullanıcı profili, bir ürün ve aynı ürün için yazılmış üç farklı "
    "reklam metni (A, B, C) verilecek. Her birini dört eksende 1-5 arası puanla:\n"
    "  - personalization: profile uyum (5 en iyi)\n"
    "  - naturalness: Türkçe doğallık (5 en iyi)\n"
    "  - product_clarity: ürün netliği (5 en iyi)\n"
    "  - creep_factor: kullanıcının duygusunu açıkça dile getirme (1 en iyi)\n"
    "ÇIKTI KURALLARI: Yalnızca geçerli JSON döndür. Markdown kod bloğu (```) "
    "kullanma, açıklama veya başlık ekleme, trailing comma kullanma. "
    'Şema: {"A":{"personalization":int,"naturalness":int,"product_clarity":int,'
    '"creep_factor":int,"reasoning":"tek cümle"},"B":{...},"C":{...}}'
)


def build_judge_prompt(persona_text, product, ads_anon):
    feats = ", ".join(product.get("key_features",[]))
    block = "\n\n".join(f"--- REKLAM {tag} ---\n{ad}" for tag, ad in ads_anon.items())
    return (f"KULLANICI PROFİLİ:\n{persona_text}\n\nÜRÜN:\nİsim: {product['name']}\n"
            f"Açıklama: {product['description']}\nÖzellikler: {feats}\n\n"
            f"REKLAMLAR:\n{block}\n\nLütfen JSON puanlamayı döndür.")


def _extract_json(raw: str) -> dict:
    """Multi-strategy JSON parsing for messy LLM outputs.
    Handles: code fences, trailing commas, surrounding prose, malformed bodies.
    Last-resort: regex-extract per-tag scores so the pipeline doesn't crash."""
    s = raw.strip()
    s = re.sub(r"^```(?:json|JSON)?\s*\n?", "", s)
    s = re.sub(r"\n?```\s*$", "", s).strip()
    try: return json.loads(s)
    except json.JSONDecodeError: pass
    i, j = s.find("{"), s.rfind("}")
    if i != -1 and j > i:
        cand = s[i:j+1]
        try: return json.loads(cand)
        except json.JSONDecodeError: pass
        fixed = re.sub(r",(\s*[}\]])", r"\1", cand)  # trailing commas
        try: return json.loads(fixed)
        except json.JSONDecodeError: pass
    out = {}
    for tag in "ABC":
        m = re.search(rf'"{tag}"\s*:\s*\{{(.+?)\}}', s, flags=re.DOTALL)
        if not m: continue
        body = m.group(1); scores = {}
        for axis in ("personalization","naturalness","product_clarity","creep_factor"):
            am = re.search(rf'"{axis}"\s*:\s*(\d+)', body)
            if am: scores[axis] = int(am.group(1))
        if len(scores) == 4:
            scores["reasoning"] = "regex-extracted (malformed JSON)"
            out[tag] = scores
    if len(out) == 3:
        return out
    raise ValueError(f"Could not extract JSON from LLM output. "
                     f"First 400 chars: {raw[:400]!r}")


def _gen_groq_json(system, user, model):
    """Groq with forced JSON output mode + lower temp — for the judge call."""
    from groq import Groq
    client = Groq(api_key=os.environ["GROQ_API_KEY"])
    last = getattr(_gen_groq, "_last_call", 0.0)
    elapsed = time.time() - last
    if elapsed < _GROQ_MIN_INTERVAL:
        time.sleep(_GROQ_MIN_INTERVAL - elapsed)
    def _call():
        return client.chat.completions.create(
            model=model,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.2, max_tokens=1500,
            response_format={"type": "json_object"},
        )
    try: resp = _call()
    except Exception as e:
        if any(s in str(e).lower() for s in ("429","rate","quota")):
            print("  groq rate-limited, sleeping 30s and retrying once…")
            time.sleep(30); resp = _call()
        else: raise
    _gen_groq._last_call = time.time()
    return resp.choices[0].message.content.strip()


def _stub_judge(persona_text, product, ads_anon, eval_lookup):
    out = {}
    for tag, ad in ads_anon.items():
        m = eval_lookup[ad]
        p = 1 + min(4, round(m["emotion_word_density"]*30 + m["persona_keyword_overlap"]))
        n = 3 + (1 if m["in_word_band"] else 0) - (1 if m["explicit_emotion_callout"] else 0)
        n = max(1, min(5, n))
        c = 1 + 2*int(m["product_name_mentioned"]) + 2*int(m["feature_mentioned"])
        cr = 5 if m["explicit_emotion_callout"] else 1
        out[tag] = {"personalization":p,"naturalness":n,"product_clarity":c,
                    "creep_factor":cr,"reasoning":"stub: derived from Layer-2 features"}
    return out


def _llm_judge(persona_text, product, ads_anon, backend, model):
    user = build_judge_prompt(persona_text, product, ads_anon)
    key = _cache_key(JUDGE_SYSTEM, user, f"judge:{backend}:{model}")
    hit = _cache_get(key)
    if hit:
        raw = hit["ad_text"]
    else:
        # Use Groq's native JSON-mode if available; otherwise generic backend.
        backend_fn = _gen_groq_json if backend == "groq" else _BACKENDS[backend]
        raw = backend_fn(JUDGE_SYSTEM, user, model)
        _cache_put(key, {"ad_text":raw,"meta":{"backend":backend,"model":model}})
    return _extract_json(raw)


by_pair = defaultdict(dict)
for a in rows:
    by_pair[(a["user_id"], a["product_id"])][a["variant"]] = a
eval_lookup = {r["ad_text"]: r for r in per_ad}
rng = random.Random(SEED); judge_results = []

for (uid, pid), variants in by_pair.items():
    if not all(v in variants for v in ("full","random","generic")): continue
    order = ["full","random","generic"]; rng.shuffle(order)
    tags = ["A","B","C"]; reveal = dict(zip(tags, order))
    ads_anon = {tag: variants[reveal[tag]]["ad_text"] for tag in tags}
    persona_text = personas_by_id[uid]["personas"]["full"]
    if BACKEND == "stub":
        scores = _stub_judge(persona_text, products_by_id[pid], ads_anon, eval_lookup)
    else:
        model_for_judge = MODEL or "llama-3.3-70b-versatile"
        scores = _llm_judge(persona_text, products_by_id[pid], ads_anon, BACKEND, model_for_judge)
    per_variant = {reveal[tag]: scores[tag] for tag in tags}
    judge_results.append({"user_id":uid,"product_id":pid,
        "true_dominant": variants["full"]["true_dominant"],
        "reveal":reveal,"scores":per_variant})

with open(OUT/"judge_results.json","w",encoding="utf-8") as f:
    json.dump(judge_results, f, ensure_ascii=False, indent=2)

# Aggregate
jrows = []
for r in judge_results:
    for v, s in r["scores"].items():
        jrows.append({"variant":v, **{k:val for k,val in s.items() if isinstance(val,(int,float))}})
jdf = pd.DataFrame(jrows)
summary = jdf.groupby("variant").agg(["mean","std"]).round(2)
summary.to_csv(OUT/"judge_summary.csv")
print("=== Per-variant judge ratings (mean ± sd, 1-5) ==="); print(summary.to_string())

print("\n=== Per-axis variant ranking ===")
means = jdf.groupby("variant").mean(numeric_only=True).round(2)
for axis in ["personalization","naturalness","product_clarity","creep_factor"]:
    asc = (axis == "creep_factor")
    ranking = means[axis].sort_values(ascending=asc)
    print(f"  {axis:18s} ({'lower better' if asc else 'higher better'}):  "
          + " > ".join(f"{v}({s})" for v,s in ranking.items()))

print(f"\n[backend={BACKEND}] Judge ratings are "
      f"{'STUB (rule-based)' if BACKEND=='stub' else 'real LLM judgments'}.")

## §13 Hyperparameter sweep — BERTurk (rubric §4)

A 2×2 grid over learning rate ∈ {2e-5, 5e-5} and batch size ∈ {16, 32} on the same train/val splits as §4. We run 3 epochs per cell (instead of 6) since the model converges fast on this dataset; total compute ≈ 4 × ~1.5 min on Colab T4 ≈ 6 min.

The point isn't to find a *better* config than §4 (BERTurk already hits 0.9949 macro-F1 there) — it's to show the model is robust to plausible hyperparameter changes and to pin a defensible recipe in the report (rubric §4 "hyperparameter comparison").

In [ ]:
# 2×2 hyperparameter sweep on BERTurk. Each cell is a 3-epoch fine-tune on
# the same splits as §4. Reports val macro-F1 per cell + heatmap.
import itertools
sweep_lrs    = [2e-5, 5e-5]
sweep_bsizes = [16, 32]
sweep_epochs = 3
sweep_results = []

for lr_v, bs_v in itertools.product(sweep_lrs, sweep_bsizes):
    print(f"--- sweep: lr={lr_v}  bs={bs_v}  epochs={sweep_epochs} ---")
    # Fresh model
    m_sw = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name, num_labels=len(EMOTION_LABELS),
        id2label=EMOTION_ID2LABEL, label2id=EMOTION_LABEL2ID,
        problem_type="single_label_classification",
    ).to(DEVICE)

    tl = DataLoader(ds_train, batch_size=bs_v, shuffle=True)
    vl = DataLoader(ds_val,   batch_size=cfg.eval_bs)

    no_decay_sw = ["bias", "LayerNorm.weight"]
    grouped_sw = [
        {"params": [p for n,p in m_sw.named_parameters() if not any(nd in n for nd in no_decay_sw)],
         "weight_decay": cfg.weight_decay},
        {"params": [p for n,p in m_sw.named_parameters() if     any(nd in n for nd in no_decay_sw)],
         "weight_decay": 0.0},
    ]
    optim_sw = AdamW(grouped_sw, lr=lr_v)
    total_steps_sw = sweep_epochs * len(tl)
    sched_sw = get_linear_schedule_with_warmup(
        optim_sw, int(cfg.warmup_frac*total_steps_sw), total_steps_sw)
    scaler_sw = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best_val = -1.0
    for ep in range(1, sweep_epochs+1):
        m_sw.train()
        for batch in tl:
            batch = {k: v.to(DEVICE, non_blocking=True) for k,v in batch.items()}
            optim_sw.zero_grad(set_to_none=True)
            if USE_AMP:
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    out = m_sw(**batch)
                scaler_sw.scale(out.loss).backward()
                scaler_sw.unscale_(optim_sw)
                torch.nn.utils.clip_grad_norm_(m_sw.parameters(), cfg.grad_clip)
                scaler_sw.step(optim_sw); scaler_sw.update()
            else:
                out = m_sw(**batch); out.loss.backward()
                torch.nn.utils.clip_grad_norm_(m_sw.parameters(), cfg.grad_clip)
                optim_sw.step()
            sched_sw.step()
        _, val_f1, _, _ = evaluate(m_sw, vl, DEVICE)
        if val_f1 > best_val: best_val = val_f1
        print(f"  epoch {ep}/{sweep_epochs}  val_macro_f1={val_f1:.4f}")
    sweep_results.append({"lr": lr_v, "batch_size": bs_v,
                          "best_val_macro_f1": float(best_val)})
    del m_sw
    if torch.cuda.is_available(): torch.cuda.empty_cache()

sweep_df = pd.DataFrame(sweep_results)
sweep_df.to_csv(OUT/"sweep_results.csv", index=False)
print("\n=== Sweep results ===")
# format lr as scientific so small floats render readably
sweep_disp = sweep_df.copy()
sweep_disp["lr"] = sweep_disp["lr"].map(lambda x: f"{x:.0e}")
sweep_disp["best_val_macro_f1"] = sweep_disp["best_val_macro_f1"].round(4)
print(sweep_disp.to_string(index=False))

# Heatmap
hm = sweep_df.pivot(index="lr", columns="batch_size",
                    values="best_val_macro_f1")
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(hm.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(hm.columns))); ax.set_xticklabels(hm.columns)
ax.set_yticks(range(len(hm.index)));   ax.set_yticklabels(hm.index)
ax.set_xlabel("batch size"); ax.set_ylabel("learning rate")
ax.set_title("BERTurk val macro-F1 — LR × batch sweep")
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        ax.text(j, i, f"{hm.values[i,j]:.4f}",
                ha="center", va="center",
                color="white" if hm.values[i,j] < hm.values.mean() else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.savefig(OUT/"sweep_heatmap.png", dpi=130, bbox_inches="tight"); plt.show()
print(f"\nSaved → {OUT}/sweep_results.csv, {OUT}/sweep_heatmap.png")

## §14 Frozen-encoder ablation (rubric §4)

Quick ablation: freeze BERTurk's encoder weights and train only a logistic head on top of the `[CLS]` representation. This isolates *how much* fine-tuning the encoder body matters versus just learning a classifier on pretrained representations.

Expectation: frozen-encoder score should be lower than full fine-tune (§4) but possibly competitive with the classical baselines (§3) — pretrained Turkish features are powerful even without task adaptation.

In [ ]:
# Frozen-encoder ablation: extract [CLS] features once, train a logistic head.
from sklearn.linear_model import LogisticRegression as _LR

@torch.no_grad()
def _cls_features(loader, bert_model, device):
    """Extract [CLS] embeddings for all rows in a loader."""
    bert_model.eval()
    feats, labels = [], []
    for batch in loader:
        labels_cpu = batch["labels"]
        ids  = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        # Use the underlying BertModel (not the classification head)
        out = bert_model.bert(input_ids=ids, attention_mask=mask)
        cls = out.last_hidden_state[:, 0, :].detach().float().cpu().numpy()
        feats.append(cls); labels.append(labels_cpu.numpy())
    return np.concatenate(feats), np.concatenate(labels)

# Use the FRESH (unfine-tuned) BERTurk for frozen features
print("Loading fresh BERTurk for frozen-feature extraction…")
m_frozen = AutoModelForSequenceClassification.from_pretrained(
    cfg.model_name, num_labels=len(EMOTION_LABELS),
    id2label=EMOTION_ID2LABEL, label2id=EMOTION_LABEL2ID,
    problem_type="single_label_classification",
).to(DEVICE)
m_frozen.eval()

train_loader_full = DataLoader(ds_train, batch_size=cfg.eval_bs)
print("Extracting [CLS] features (train)…")
X_tr_fr, y_tr_fr = _cls_features(train_loader_full, m_frozen, DEVICE)
print("Extracting [CLS] features (test)…")
X_te_fr, y_te_fr = _cls_features(test_loader,       m_frozen, DEVICE)
print(f"feature dim: {X_tr_fr.shape[1]}")

# Train a simple logistic head on frozen features
clf_fr = _LR(C=1.0, max_iter=2000, n_jobs=-1)
clf_fr.fit(X_tr_fr, y_tr_fr)
y_pred_fr = clf_fr.predict(X_te_fr)
acc_fr = accuracy_score(y_te_fr, y_pred_fr)
f1_fr  = f1_score(y_te_fr, y_pred_fr, average="macro")
print(f"\n=== FROZEN BERTurk + LogReg head ===")
print(f"test acc={acc_fr:.4f}  macro_f1={f1_fr:.4f}")
print(classification_report(y_te_fr, y_pred_fr,
      target_names=EMOTION_LABELS, digits=4))

# Update the master comparison
print("\n" + "="*60)
print("EXTENDED COMPARISON (with frozen-encoder ablation)")
print("="*60)
ext = pd.DataFrame([
    {"model": "TF-IDF + LogReg",         **baseline_results["logreg"]},
    {"model": "TF-IDF + Linear SVM",     **baseline_results["svm"]},
    {"model": "Frozen BERTurk + LogReg", "acc": float(acc_fr), "macro_f1": float(f1_fr)},
    {"model": "BERTurk (full FT)",       "acc": float(test_acc),   "macro_f1": float(test_f1)},
    {"model": "XLM-R (full FT)",         "acc": float(test_acc_x), "macro_f1": float(test_f1_x)},
])
print(ext[["model","acc","macro_f1"]].round(4).to_string(index=False))
ext.to_csv(OUT/"model_comparison_extended.csv", index=False)
del m_frozen
if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f"\nSaved → {OUT}/model_comparison_extended.csv")

## §15 Robustness — character-level typo injection (rubric §4)

How does test-set macro-F1 degrade when we inject character-level noise into the input? We swap a fraction of characters for each tweet at noise rates 0%, 5%, 10%, 20%, 30% and re-evaluate the fine-tuned BERTurk on each level.

This isolates the model's tolerance to misspellings, OCR-style errors, and informal writing — directly relevant for tweet-domain deployment where the input is often noisy.

In [ ]:
# Character-level noise injection robustness test.
import string
TURKISH_CHARS = "abcçdefgğhıijklmnoöprsştuüvyz"
rng_rob = random.Random(SEED)

def inject_noise(text: str, p: float, rng) -> str:
    """Replace each character with a random Turkish char with probability p."""
    if p <= 0: return text
    out = []
    for ch in text:
        if ch.isalpha() and rng.random() < p:
            out.append(rng.choice(TURKISH_CHARS))
        else:
            out.append(ch)
    return "".join(out)

# Reload best fine-tuned BERTurk
print("Reloading fine-tuned BERTurk for robustness test…")
m_rob = AutoModelForSequenceClassification.from_pretrained(
    cfg.model_name, num_labels=len(EMOTION_LABELS),
    id2label=EMOTION_ID2LABEL, label2id=EMOTION_LABEL2ID,
    problem_type="single_label_classification",
).to(DEVICE)
m_rob.load_state_dict(torch.load(best_path, map_location=DEVICE))
m_rob.eval()

noise_rates = [0.0, 0.05, 0.10, 0.20, 0.30]
robustness_results = []

for p in noise_rates:
    # Build a noisy version of the test set
    noisy_texts = [inject_noise(t, p, rng_rob) for t in df_test["text"].values]
    ds_noisy = EmotionDS(noisy_texts, df_test["label_id"].values, tok, cfg.max_length)
    loader_noisy = DataLoader(ds_noisy, batch_size=cfg.eval_bs)
    _, f1_n, _, _ = evaluate(m_rob, loader_noisy, DEVICE)
    robustness_results.append({"noise_rate": p, "test_macro_f1": float(f1_n)})
    print(f"noise={p*100:>5.0f}%  test_macro_f1={f1_n:.4f}  Δ={f1_n-test_f1:+.4f}")

rob_df = pd.DataFrame(robustness_results)
rob_df.to_csv(OUT/"robustness_results.csv", index=False)

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rob_df["noise_rate"]*100, rob_df["test_macro_f1"],
        marker="o", color="#E45756", linewidth=2)
ax.axhline(test_f1, ls="--", color="grey", alpha=0.6,
           label=f"clean test = {test_f1:.4f}")
ax.set_xlabel("character-noise rate (%)"); ax.set_ylabel("test macro F1")
ax.set_title("BERTurk robustness — character-level noise injection")
ax.set_ylim(0, 1.02); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig(OUT/"robustness_curve.png", dpi=130, bbox_inches="tight"); plt.show()
del m_rob
if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f"\nSaved → {OUT}/robustness_results.csv, {OUT}/robustness_curve.png")

## Done

Outputs are in `OUT_ROOT` (default `./outputs`):

```
outputs/
├── berturk_emotion/{pytorch_model.bin, tokenizer/, history.png, confusion_matrix.png, test_metrics.json}
├── xlmr_emotion/{pytorch_model.bin, confusion_matrix.png, test_metrics.json}
├── baselines/{results.json, confusion_matrices.png}
├── model_comparison.csv, model_comparison.png
├── users.json, user_vectors.json, personas.json, products.json
├── ads_results.{json,csv}, ads_cache/
├── eval_results.json, eval_summary.csv, eval_diff_pairs.csv
├── judge_results.json, judge_summary.csv
├── sweep_results.csv, sweep_heatmap.png             (§13)
├── model_comparison_extended.csv                    (§14)
└── robustness_results.csv, robustness_curve.png     (§15)
```

**LLM backend recap:**
- **Groq only** (Llama-3.3-70B-Versatile) when `GROQ_API_KEY` is configured.
- Falls back to the deterministic stub backend when no key is set so the notebook still runs end-to-end.
- Disk cache (`ads_cache/`) keys on `groq:llama-3.3-70b-versatile` so re-runs are free.

**Drive persistence:** the §0 cell mounts `/content/drive/MyDrive/outputs/` and all subsequent saves (trained BERTurk + XLM-R checkpoints, ads, eval CSVs) land there, so they survive Colab session timeouts.

**Total runtime on Colab T4 (one-shot with Groq):** ~10 min for §0–§7, ~3-5 min for §4 training, ~2-3 min for §11 (paced at 2.5s/call to stay under Groq free-tier 30 RPM), ~1-2 min for §12. Re-runs use the cache and complete in seconds.